# **Install requirements**

In [ ]:
!pip install comet_ml > /dev/null 2>&1
import comet_ml
# TODO: ENTER YOUR API KEY HERE!! instructions above
COMET_API_KEY = "xNV7ssLfwo53ENbdFvirdm7zV"
# Import Tensorflow 2.0
import tensorflow as tf

# Download and import the MIT Introduction to Deep Learning package
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

# Import all remaining packages
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write
!apt-get install abcmidi timidity > /dev/null 2>&1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **Dataset**

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

In [ ]:
# Convert the ABC notation to audio file and listen to it
mdl.lab1.play_song(example_song)

In [ ]:
# Join our list of song strings into a single string containing all songs
songs_joined = "\n\n".join(songs)

# Find all unique characters in the joined string
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

## **Process the dataset for the learning task**

### Vectorize the text

* We'll generate two lookup tables: one that maps characters to numbers, and a second that maps numbers back to characters



In [ ]:
### Define numerical representation of text ###

# Create a mapping from character to unique index.
# For example, to get the index of the character "d",
#   we can evaluate `char2idx["d"]`.
char2idx = {u:i for i, u in enumerate(vocab)}

# Create a mapping from indices to characters. This is
#   the inverse of char2idx and allows us to convert back
#   from unique index to the character in our vocabulary.
idx2char = np.array(vocab)

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
### Vectorize the songs string ###

# convert the all songs string to a vectorized (i.e., numeric) representation
def vectorize_string(string):
    vectorized_output = np.array([char2idx[str] for str in string])
    # array with N elements, where N is the number of characters in the string
    return vectorized_output

vectorized_songs = vectorize_string(songs_joined)

In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# check that vectorized_songs is a numpy array
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Create training examples and targets
* We'll break the text into chunks of seq_length+1. Suppose seq_length is 4 and our text is "Hello". Then, our input sequence is "Hell" and the target sequence is "ello"

In [ ]:
### Batch definition to create training examples ###

def get_batch(vectorized_songs, seq_length, batch_size):
  # the length of the vectorized songs string
  n = vectorized_songs.shape[0] - 1
  # randomly choose the starting indices for the examples in the training batch
  idx = np.random.choice(n-seq_length, batch_size)

  input_batch = [vectorized_songs[i:i+seq_length] for i in idx ]

  output_batch = [vectorized_songs[i+1: i+seq_length + 1] for i in idx]

  # x_batch, y_batch provide the true inputs and targets for network training
  x_batch = np.reshape(input_batch, [batch_size, seq_length])
  y_batch = np.reshape(output_batch, [batch_size, seq_length])
  return x_batch, y_batch


# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(np.squeeze(x_batch), np.squeeze(y_batch))):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

# **The Recurrent Neural Network (RNN) model**

Three layers are used:

* tf.keras.layers.Embedding: This is the input layer, consisting of a trainable lookup table that maps the numbers of each character to a vector with embedding_dim dimensions.
* tf.keras.layers.LSTM: Our LSTM network, with size units=rnn_units.
* tf.keras.layers.Dense: The output layer, with vocab_size outputs.

### Defining the RNN model

In [ ]:
def LSTM(rnn_units):
  return tf.keras.layers.LSTM(
    rnn_units,
    return_sequences=True,
    recurrent_initializer='glorot_uniform',
    recurrent_activation='sigmoid',
    stateful=True,
  )

In [ ]:
### Defining the RNN Model ###

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
    # Layer 1: Embedding layer to transform indices into dense vectors of a fixed embedding size
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    # Layer 2: LSTM with `rnn_units` number of units.
    LSTM(rnn_units),
    # Layer 3: Dense (fully-connected) layer that transforms the LSTM output into the vocabulary size.
    tf.keras.layers.Dense(vocab_size)
    ])
    return model

# Build a simple model with default hyperparameters. You will get the
#   chance to change these later.
model = build_model(len(vocab), embedding_dim=256, rnn_units=1024, batch_size=32)
model.build(tf.TensorShape([32, 100]))  # [batch_size, sequence_length]

### Test out the RNN model

In [ ]:
model.summary()

In [ ]:
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

### Predictions from the untrained model

In [ ]:
sampled_indices = tf.random.categorical(pred[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()
sampled_indices

In [ ]:
print("Input: \n", repr("".join(idx2char[x[0]])))
print()
print("Next Char Predictions: \n", repr("".join(idx2char[sampled_indices])))

# **Training the model: loss and training operations**

In [ ]:
### Defining the loss function ###

def compute_loss(labels, logits):
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
    return loss

example_batch_loss = compute_loss(y, pred)

print("Prediction shape: ", pred.shape, " # (batch_size, sequence_length, vocab_size)")
print("scalar_loss:      ", example_batch_loss.numpy().mean())

In [ ]:
### Hyperparameter setting and optimization ###

vocab_size = len(vocab)

# Model parameters:
params = dict(
  num_training_iterations = 200,  # Increase this to train longer
  batch_size = 8,  # Experiment between 1 and 64
  seq_length = 100,  # Experiment between 50 and 500
  learning_rate = 5e-3,  # Experiment between 1e-5 and 1e-1
  embedding_dim = 256,
  rnn_units = 1024,  # Experiment between 1 and 2048
)

# Checkpoint location:
checkpoint_dir ='/content/drive/MyDrive/Music_Generation/training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt.weights.h5")
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
### Create a Comet experiment to track our training run ###

def create_experiment():
  # end any prior experiments
  if 'experiment' in locals():
    experiment.end()

  # initiate the comet experiment for tracking
  experiment = comet_ml.Experiment(
                  api_key=COMET_API_KEY,
                  project_name="6S191_Lab1_Part2")
  # log our hyperparameters, defined above, to the experiment
  for param, value in params.items():
    experiment.log_parameter(param, value)
  experiment.flush()

  return experiment

In [ ]:
### Resume training from the existing checkpoint, appending to a persisted loss history ###
import json

loss_history_path = "loss_history.json"

# Tune this and re-run the cell to train longer, a bit at a time.
additional_iterations = 200

if os.path.exists(loss_history_path):
    with open(loss_history_path, "r") as f:
        saved = json.load(f)
    history = saved["loss"]
    total_iterations_so_far = saved["total_iterations"]
    print(f"Resuming: {total_iterations_so_far} iterations already completed, "
          f"{len(history)} loss values loaded from {loss_history_path}")
else:
    history = []
    total_iterations_so_far = 200
    print("No existing loss history found; starting fresh.")

# Rebuild the model at training batch size and load existing weights if present.
# Embedding/LSTM/Dense weights are batch-size-independent, exactly as the
# checkpoint-restore cell already relies on when restoring for generation at batch_size=1.
model = build_model(len(vocab), params["embedding_dim"], params["rnn_units"], params["batch_size"])
model.build(tf.TensorShape([params["batch_size"], None]))
if os.path.exists(checkpoint_prefix):
    model.load_weights(checkpoint_prefix)
    print(f"Loaded existing weights from {checkpoint_prefix}")
else:
    print("No checkpoint found; training from randomly-initialized weights.")

optimizer = tf.keras.optimizers.Adam(params["learning_rate"])

@tf.function
def train_step(x, y):
  with tf.GradientTape() as tape:
    y_hat = model(x)
    loss = compute_loss(y, y_hat)
  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  return loss

plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations (cumulative)', ylabel='Loss')
experiment = create_experiment()

if hasattr(tqdm, '_instances'): tqdm._instances.clear()
for iter in tqdm(range(additional_iterations)):
  x_batch, y_batch = get_batch(vectorized_songs, params["seq_length"], params["batch_size"])
  loss = train_step(x_batch, y_batch)
  experiment.log_metric("loss", loss.numpy().mean(), step=total_iterations_so_far + iter)
  history.append(float(loss.numpy().mean()))
  plotter.plot(history)
  if iter % 100 == 0:
    model.save_weights(checkpoint_prefix)

model.save_weights(checkpoint_prefix)
experiment.flush()

total_iterations_so_far += additional_iterations

# Keep a separate, uniquely-named copy of this checkpoint so past iteration
# counts stay available for comparison -- checkpoint_prefix itself always
# holds only the latest weights, since it gets overwritten on every run.
import shutil
versioned_checkpoint = os.path.join(
    checkpoint_dir, f"ckpt_{total_iterations_so_far}iters.weights.h5"
)
shutil.copy(checkpoint_prefix, versioned_checkpoint)
print(f"Also saved a snapshot to {versioned_checkpoint}")

with open(loss_history_path, "w") as f:
    json.dump({"loss": history, "total_iterations": total_iterations_so_far}, f)

print(f"Saved {len(history)} cumulative loss values "
      f"({total_iterations_so_far} total training iterations) to {loss_history_path}")


# Generate music using the RNN model

In [ ]:
# Restore the latest checkpoint

model = build_model(vocab_size, params["embedding_dim"], params["rnn_units"], batch_size=1)
# Restore the model weights for the last checkpoint after training
model.build(tf.TensorShape([1, None]))
model.load_weights(checkpoint_prefix)

model.summary()

In [ ]:
### Prediction of a generated song ###
def generate_text(model, start_string, generation_length=1000):
  # Evaluation step (generating ABC text using the learned RNN model)

  input_eval = [char2idx[str] for str in start_string]
  input_eval = tf.expand_dims(input_eval, 0)

  # Empty string to store our results
  text_generated = []

  # Here batch size == 1
  for layer in model.layers:
    if hasattr(layer, "reset_states"):
      layer.reset_states()

  tqdm._instances.clear()

  for i in tqdm(range(generation_length)):
    predictions = model(input_eval)
    # Remove the batch dimension
    predictions = tf.squeeze(predictions, 0)
    predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

    # Pass the prediction along with the previous hidden state
    #   as the next inputs to the model
    input_eval = tf.expand_dims([predicted_id], 0)

    text_generated.append(idx2char[predicted_id])

  return (start_string + ''.join(text_generated))

In [ ]:
generated_text = generate_text(model, start_string="X", generation_length=1000)

## Play back the generated music!

*Audio player output stripped to keep the notebook lightweight; listen here:* [example_song.wav](example_song.wav)

In [ ]:
### Play back generated songs ###

generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
  # Synthesize the waveform from a song
  waveform = mdl.lab1.play_song(song)

  # If its a valid song (correct syntax), lets play it!
  if waveform:
    print("Generated song", i)
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # save your song to the Comet interface -- you can access it there
    experiment.log_asset(wav_file_path)
# when done, end the comet experiment
experiment.end()

*Audio player outputs stripped to keep the notebook lightweight; listen here:* [output_1.wav](output_1.wav) · [output_2.wav](output_2.wav)